In [ ]:
from pathlib import Path
import re

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from glob import glob
import numpy as np

In [ ]:
PROJECT_ROOT = Path("/mnt/local/data/ksozykin/src/ESdB-2/")

TASK_MAIN_METRICS = {
    "classification": ["clf", "age"],
    "regression": ["reg", "reg_mentions"],
    "forecasting": ["forecast"],
    "anomaly": ["anomaly"],
}

EPOCH_DIR_RE = re.compile(r"^epoch_(?P<epoch>\d+)(?:\((?P<version>\d+)\))?$")
TARGET_METRIC_RE = re.compile(
    r"^target__(?P<metric>.+?)__.*__validator_seed_(?P<seed>\d+)$"
)

In [ ]:
def parse_epoch_dir_name(path):
    match = EPOCH_DIR_RE.match(path.name)
    if match is None:
        return None

    epoch = int(match.group("epoch"))
    version = int(match.group("version") or 0)
    return epoch, version


def latest_epoch_results(reval_dir):
    latest_by_epoch = {}

    for epoch_dir in reval_dir.iterdir():
        parsed = parse_epoch_dir_name(epoch_dir)
        if parsed is None:
            continue

        results_path = epoch_dir / "results.csv"
        if not results_path.exists():
            continue

        epoch, version = parsed
        previous = latest_by_epoch.get(epoch)
        if previous is None or version > previous[0]:
            latest_by_epoch[epoch] = (version, results_path)

    return [
        latest_by_epoch[epoch][1]
        for epoch in sorted(latest_by_epoch)
    ]


def task_name_from_dir(task_dir):
    return task_dir.name.removeprefix("best_")


def as_list(value):
    if isinstance(value, str):
        return [value]
    return list(value)


def read_results_values(results_path, value_col="mean"):
    df = pd.read_csv(results_path, index_col=0)
    if value_col not in df.columns:
        value_col = df.columns[0]

    return pd.to_numeric(df[value_col], errors="coerce")


def choose_main_metric(metric_names, task_name, main_metric_map=None):
    main_metric_map = main_metric_map or TASK_MAIN_METRICS
    candidates = main_metric_map.get(task_name, [])

    for metric_name in candidates:
        if metric_name in metric_names:
            return metric_name

    return None


def parse_results_csv(results_path, task_name, value_col="mean", main_metric_map=None):
    values = read_results_values(results_path, value_col=value_col)
    epoch, version = parse_epoch_dir_name(results_path.parent)
    metrics = {}

    row = {
        "task": task_name,
        "epoch": epoch,
    }

    for name, value in values.items():
        metric_match = TARGET_METRIC_RE.match(name)
        if metric_match is not None:
            metric_name = metric_match.group("metric")
            seed = int(metric_match.group("seed")) + 1
            metrics.setdefault(metric_name, {})[seed] = value
            continue

        row[name] = value

    for metric_name, seed_values in sorted(metrics.items()):
        metric_series = pd.Series(seed_values, dtype="float64").sort_index()
        for seed, value in metric_series.items():
            row[f"{metric_name}_seed{seed}"] = value

        row[f"{metric_name}_mean"] = metric_series.mean()
        row[f"{metric_name}_max"] = metric_series.max()

    main_metric = choose_main_metric(metrics.keys(), task_name, main_metric_map)
    row["main_metric"] = main_metric
    if main_metric is not None:
        main_series = pd.Series(metrics[main_metric], dtype="float64").sort_index()
        for seed in range(1, 4):
            row[f"main_seed{seed}"] = main_series.get(seed)

        row["main_mean"] = main_series.mean()
        row["main_max"] = main_series.max()

    return row


def merge_metric_rows(base_row, new_row):
    for key, value in new_row.items():
        if key not in base_row:
            base_row[key] = value

    return base_row


def build_task_df(task_dir, reval_names=("reval_sample",), value_col="mean", main_metric_map=None):
    task_dir = Path(task_dir)
    task_name = task_name_from_dir(task_dir)
    rows_by_epoch = {}

    for reval_name in as_list(reval_names):
        reval_dir = task_dir / reval_name
        if not reval_dir.exists():
            continue

        for results_path in latest_epoch_results(reval_dir):
            row = parse_results_csv(
                results_path,
                task_name=task_name,
                value_col=value_col,
                main_metric_map=main_metric_map,
            )
            row["source_revals"] = reval_name
            epoch = row["epoch"]

            if epoch in rows_by_epoch:
                rows_by_epoch[epoch]["source_revals"] += f",{reval_name}"
                merge_metric_rows(rows_by_epoch[epoch], row)
            else:
                rows_by_epoch[epoch] = row

    if not rows_by_epoch:
        return pd.DataFrame()

    return pd.DataFrame(rows_by_epoch.values()).sort_values("epoch").reset_index(drop=True)


def build_task_dfs(tests_dir, reval_names=("reval_sample",), value_col="mean", main_metric_map=None):
    tests_dir = Path(tests_dir)
    task_dfs = {}

    for task_dir in sorted(tests_dir.glob("*")):
        if not task_dir.is_dir():
            continue

        task_name = task_name_from_dir(task_dir)
        task_dfs[task_name] = build_task_df(
            task_dir,
            reval_names=reval_names,
            value_col=value_col,
            main_metric_map=main_metric_map,
        )

    return task_dfs


In [ ]:
# REVAL_NAMES = ["reval_sample", "reval_effrank"]
REVAL_NAMES = ["reval"]
DATASET = "twitter"
METHOD = "ntp_gpt"
TESTS_DIR = PROJECT_ROOT / "log" / "full" / DATASET / METHOD.upper() / "tests"


In [ ]:
#ls $TESTS_DIR

In [ ]:
task_dfs = build_task_dfs(TESTS_DIR, reval_names=REVAL_NAMES)

classification_df = task_dfs.get("classification", pd.DataFrame())
regression_df = task_dfs.get("reg", pd.DataFrame())
forecasting_df = task_dfs.get("forecast", pd.DataFrame())
anomaly_df = task_dfs.get("anomaly", pd.DataFrame())

{task: df.shape for task, df in task_dfs.items()}


In [ ]:

clean_cols = {
    "epoch": "epoch",
    "loss": "loos",
    "train_loss": "train_loss",
    #"train_jepa_loss": "val_loss",
    #"reg_mean": "reg",
     "reg_mentions_mean": "reg",
    "clf_mean": "clf",
    "anomaly_mean": "anom",
    "forecast_mean": "forecast",
    "embedding__train__global__effective_rank": "eff_rank_train_global",
    # "embedding__test__global__effective_rank": "eff_rank_test_global",
    "embedding__train__shift__effective_rank": "eff_rank_train_shift",
    # "embedding__test__shift__effective_rank": "eff_rank_test_shift",
    # "embedding__train__global__anisotropy": "anis_train_global",
    # "embedding__test__global__anisotropy": "anis_test_global",
    # "embedding__train__shift__anisotropy": "anis_train_shift",
    # "embedding__test__shift__anisotropy": "anis_test_shift",
}


def make_clean_df(df, clean_cols):
    cols = [col for col in clean_cols if col in df.columns]
    return df[cols].rename(columns=clean_cols).set_index("epoch")


clean_clf = make_clean_df(classification_df, clean_cols)
clean_reg = make_clean_df(regression_df, clean_cols)
#clean_forecast = make_clean_df(forecasting_df, clean_cols)
clean_anom = make_clean_df(anomaly_df, clean_cols)


In [ ]:
P_VALUE_WARN_THRESHOLD = 0.05

def corr_and_pvalue_matrices(df, method):
    corr_fn = spearmanr if method == "spearman" else pearsonr
    cols = df.columns
    corr = pd.DataFrame(1.0, index=cols, columns=cols)
    p_values = pd.DataFrame(0.0, index=cols, columns=cols)

    for left_idx, left_col in enumerate(cols):
        for right_col in cols[left_idx + 1:]:
            pair = df[[left_col, right_col]].dropna()
            corr_value, p_value = corr_fn(pair[left_col], pair[right_col])
            corr.loc[left_col, right_col] = corr_value
            corr.loc[right_col, left_col] = corr_value
            p_values.loc[left_col, right_col] = p_value
            p_values.loc[right_col, left_col] = p_value

    return corr, p_values


def plot_corr_heatmaps(clean_dfs, method="spearman", p_value_threshold=P_VALUE_WARN_THRESHOLD):
    if method not in {"spearman", "pearson"}:
        raise ValueError('method must be "spearman" or "pearson"')

    for title, df in clean_dfs.items():
        corr, p_values = corr_and_pvalue_matrices(df, method)

        plt.figure(figsize=(11, 8), dpi=300)
        ax = sns.heatmap(
            corr,
            cmap="coolwarm",
            vmin=-1,
            vmax=1,
            cbar=True,
            linewidths=0.5,
            linecolor="white",
        )
        
        # Ручное добавление аннотаций
        n = len(corr)
        for i in range(n):
            for j in range(n):
                corr_val = corr.iloc[i, j]
                p_val = p_values.iloc[i, j]
                
                # На диагонали - только корреляция
                if i == j:
                    text = f"{corr_val:.2f}"
                # Если p-value >= threshold (незначимо) - показываем и корреляцию, и p-value
                elif p_val >= p_value_threshold:
                    text = f"{corr_val:.2f}\np={p_val:.2g}"
                # Если p-value < threshold (значимо) - только корреляция
                else:
                    text = f"{corr_val:.2f}"
                
                # Выбираем цвет текста в зависимости от фона
                color = 'white' if abs(corr_val) > 0.5 else 'black'
                ax.text(j + 0.5, i + 0.5, text, 
                       ha='center', va='center', 
                       color=color, fontsize=8)
        
        ax.set_title(f"{title}: {method.capitalize()} corr")
        plt.tight_layout()
        plt.show()


clean_dfs = {
    "Classification based": clean_clf,
    "Regression based": clean_reg,
    #"Forecasting based": clean_forecast,
    "Anomaly based": clean_anom,
}



In [ ]:
plot_corr_heatmaps(clean_dfs, method="spearman")

In [ ]:
assert

In [ ]:
assert

In [ ]:
def plot_epoch_metric_points(clean_dfs, epoch_name="epoch"):
    for title, df in clean_dfs.items():
        plot_df = (
            df
            .rename_axis(epoch_name)
            .reset_index()
            .melt(id_vars=epoch_name, var_name="metric", value_name="value")
            .dropna()
        )

        plt.figure(figsize=(9, 5), dpi=300)
        ax = sns.lineplot(
            data=plot_df,
            x=epoch_name,
            y="value",
            hue="metric",
            style="metric",
            markers=True,
            dashes=False,
            linewidth=2,
            markersize=7,
        )
        ax.set_title(f"{title}: loss and metrics by epoch")
        ax.set_xlabel(epoch_name)
        ax.set_ylabel("value")
        ax.grid(True, alpha=0.25)
        ax.legend(title="metric", bbox_to_anchor=(1.02, 1), loc="upper left")
        sns.despine()
        plt.tight_layout()
        plt.show()


In [ ]:
plot_epoch_metric_points(clean_dfs)

In [ ]:
def plot_epoch_metric_points_clipped(clean_dfs, epoch_name="epoch", y_min=-1, y_max=1):
    for title, df in clean_dfs.items():
        plot_df = (
            df
            .rename_axis(epoch_name)
            .reset_index()
            .melt(id_vars=epoch_name, var_name="metric", value_name="value")
            .dropna()
        )
        plot_df["clipped_value"] = plot_df["value"].clip(y_min, y_max)
        plot_df["is_clipped_low"] = plot_df["value"] < y_min
        plot_df["is_clipped_high"] = plot_df["value"] > y_max

        plt.figure(figsize=(9, 5), dpi=300)
        ax = sns.lineplot(
            data=plot_df,
            x=epoch_name,
            y="clipped_value",
            hue="metric",
            style="metric",
            markers=True,
            dashes=False,
            linewidth=2,
            markersize=7,
        )

        clipped_low = plot_df[plot_df["is_clipped_low"]]
        clipped_high = plot_df[plot_df["is_clipped_high"]]

        if not clipped_low.empty:
            sns.scatterplot(
                data=clipped_low,
                x=epoch_name,
                y="clipped_value",
                hue="metric",
                marker="v",
                s=120,
                edgecolor="black",
                linewidth=0.8,
                legend=False,
                ax=ax,
            )

        if not clipped_high.empty:
            sns.scatterplot(
                data=clipped_high,
                x=epoch_name,
                y="clipped_value",
                hue="metric",
                marker="^",
                s=120,
                edgecolor="black",
                linewidth=0.8,
                legend=False,
                ax=ax,
            )

        for _, row in plot_df[plot_df["is_clipped_low"] | plot_df["is_clipped_high"]].iterrows():
            # y_offset = 15 if row["is_clipped_low"] else 10
            va = "top" if row["is_clipped_low"] else "bottom"
            ax.annotate(
                f'{row["value"]:.2g}',
                xy=(row[epoch_name], row["clipped_value"]),
                xytext=(15, 15),
                textcoords="offset points",
                ha="center",
                va=va,
                fontsize=8,
                color="black",
            )

        ax.axhline(y_min, color="black", linestyle="--", linewidth=0.8, alpha=0.4)
        ax.axhline(y_max, color="black", linestyle="--", linewidth=0.8, alpha=0.4)
        ax.set_ylim(y_min, y_max)
        ax.set_title(f"{title}: loss and metrics by epoch, clipped to [{y_min}, {y_max}]")
        ax.set_xlabel(epoch_name)
        ax.set_ylabel("value")
        # ax.set_yscale('log')
        ax.grid(True, alpha=0.25)
        ax.legend(title="metric", bbox_to_anchor=(1.02, 1), loc="upper left")
        sns.despine()
        plt.tight_layout()
        plt.show()


plot_epoch_metric_points_clipped(clean_dfs, y_min=-1, y_max=1)


In [100]:
import traceback
from pathlib import Path
import re
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from glob import glob
import numpy as np

D = [Path(g).name for g in glob('../../log/full/*')]
#D = ["zvuk","alpha"]
full_name = {'reg' : 'reg', 'cl' : 'classification', 'fore' : 'forecast', 'anom' : 'anomaly'}
for d in D:
    for m in ['ntp_gru', 'ntp_gpt']:
        for t in ['reg', 'cl', 'fore', 'anom']:
            try:
                task = full_name[t]
                dfs_paths = glob(f'../../log/full/{d}/{m.upper()}/tests/{task}/results.csv')
                dfs =  [pd.read_csv(path).set_index('Unnamed: 0').T for path in dfs_paths]
                
                mcols_seed = [col for col in dfs[0].columns if '_seed_' in col]
                mcols = [col for col in mcols_seed if ('clf' if t == 'cl' else t) in col]
                if mcols == []:
                    mcols = [col for col in mcols_seed if 'global' in col]
                max_metric =  np.max([df[mcols].T['mean'].mean() for df in dfs])
                print(f"{d} {t} {m} {max_metric:.5f}")
            except:
                traceback.print_exc()
        print()

taobao reg ntp_gru 0.04861
taobao cl ntp_gru 0.67608
taobao fore ntp_gru 0.04899

taobao reg ntp_gpt -0.46519
taobao cl ntp_gpt 0.64148
taobao fore ntp_gpt 0.05236
taobao anom ntp_gpt 0.84088

favorita reg ntp_gru -1.15228
favorita cl ntp_gru 0.63889
favorita fore ntp_gru -1.37757
favorita anom ntp_gru 1.00000

favorita reg ntp_gpt -0.78092
favorita cl ntp_gpt 0.69444
favorita fore ntp_gpt -1.12857
favorita anom ntp_gpt 0.83333

age reg ntp_gru 0.46343
age cl ntp_gru 0.68302
age fore ntp_gru 0.41078
age anom ntp_gru 0.74536

age reg ntp_gpt 0.44006
age cl ntp_gpt 0.65410
age fore ntp_gpt 0.38898
age anom ntp_gpt 0.70139

zvuk reg ntp_gru 0.13059
zvuk cl ntp_gru 0.47037
zvuk fore ntp_gru -0.47273
zvuk anom ntp_gru 0.89457

zvuk reg ntp_gpt 0.11593
zvuk cl ntp_gpt 0.47925
zvuk fore ntp_gpt 0.01565
zvuk anom ntp_gpt 0.89868

ett fore ntp_gru -0.00960

ett fore ntp_gpt -0.00699

alpha reg ntp_gru 0.70962
alpha cl ntp_gru 0.57999
alpha fore ntp_gru 0.05655
alpha anom ntp_gru 0.70458

alpha 

Traceback (most recent call last):
  File "/tmp/ipykernel_819869/3291769032.py", line 23, in <module>
    mcols_seed = [col for col in dfs[0].columns if '_seed_' in col]
IndexError: list index out of range
Traceback (most recent call last):
  File "/tmp/ipykernel_819869/3291769032.py", line 23, in <module>
    mcols_seed = [col for col in dfs[0].columns if '_seed_' in col]
IndexError: list index out of range
Traceback (most recent call last):
  File "/tmp/ipykernel_819869/3291769032.py", line 23, in <module>
    mcols_seed = [col for col in dfs[0].columns if '_seed_' in col]
IndexError: list index out of range
Traceback (most recent call last):
  File "/tmp/ipykernel_819869/3291769032.py", line 23, in <module>
    mcols_seed = [col for col in dfs[0].columns if '_seed_' in col]
IndexError: list index out of range
Traceback (most recent call last):
  File "/tmp/ipykernel_819869/3291769032.py", line 23, in <module>
    mcols_seed = [col for col in dfs[0].columns if '_seed_' in col]
IndexE

In [127]:
import traceback
from pathlib import Path
import re
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from glob import glob
import numpy as np
root = "/mnt/local/data/ksozykin/src/ESdB-simclr/log/full/"
D = [Path(g).name for g in glob(f'{root}/*')]
#D = ["twitter", "yambda" , "30music"]
full_name = {'reg' : 'best_regression', 'cl' : 'best_classification', 'fore' : 'best_forecast', 'anom' : 'best_anomaly'}
for d in D:
    for m in ['SimCLR',]:
        for t in ['reg', 'cl', 'fore', 'anom']:
            try:
                task = full_name[t]
                dfs_paths = glob(f'{root}/{d}/{m}/tests/{task}/results.csv')
                dfs =  [pd.read_csv(path).set_index('Unnamed: 0').T for path in dfs_paths]
                
                mcols_seed = [col for col in dfs[0].columns if '_seed_' in col]
                mcols = [col for col in mcols_seed if ('clf' if t == 'cl' else t) in col]
                if mcols == []:
                    mcols = [col for col in mcols_seed if 'global' in col]
                max_metric =  np.max([df[mcols].T['mean'].mean() for df in dfs])
                print(f"{d} {t} {m} {max_metric:.5f}")
            except:
                traceback.print_exc()
        print()

taobao reg SimCLR 0.25666
taobao cl SimCLR 0.67523
taobao fore SimCLR 0.01178
taobao anom SimCLR 0.91864

twitter reg SimCLR 0.59815
twitter cl SimCLR 0.72384
twitter anom SimCLR 0.99922

favorita reg SimCLR -1.42084
favorita cl SimCLR 0.60417
favorita fore SimCLR -1.77005
favorita anom SimCLR 0.83333

age reg SimCLR 0.50568
age cl SimCLR 0.68976
age fore SimCLR 0.34375
age anom SimCLR 0.79995

ett fore SimCLR 0.02780

electric_devices cl SimCLR 0.83053
electric_devices fore SimCLR 0.52079

30music reg SimCLR 0.03016
30music cl SimCLR 0.68320
30music fore SimCLR 0.04208
30music anom SimCLR 0.84023

yambda reg SimCLR 0.72768
yambda cl SimCLR 0.70797
yambda fore SimCLR 0.17696
yambda anom SimCLR 0.99912

rossman reg SimCLR 0.95461
rossman cl SimCLR 0.69940
rossman fore SimCLR 0.03341
rossman anom SimCLR 0.88967



Traceback (most recent call last):
  File "/tmp/ipykernel_819869/1493380415.py", line 23, in <module>
    mcols_seed = [col for col in dfs[0].columns if '_seed_' in col]
IndexError: list index out of range
Traceback (most recent call last):
  File "/tmp/ipykernel_819869/1493380415.py", line 23, in <module>
    mcols_seed = [col for col in dfs[0].columns if '_seed_' in col]
IndexError: list index out of range
Traceback (most recent call last):
  File "/tmp/ipykernel_819869/1493380415.py", line 23, in <module>
    mcols_seed = [col for col in dfs[0].columns if '_seed_' in col]
IndexError: list index out of range
Traceback (most recent call last):
  File "/tmp/ipykernel_819869/1493380415.py", line 23, in <module>
    mcols_seed = [col for col in dfs[0].columns if '_seed_' in col]
IndexError: list index out of range
Traceback (most recent call last):
  File "/tmp/ipykernel_819869/1493380415.py", line 23, in <module>
    mcols_seed = [col for col in dfs[0].columns if '_seed_' in col]
IndexE

In [110]:
D

['full']